# Parameter visualizations

Interactive (Plotly) visualisations of the knobs that shape training, imported **live** from the
modules training uses so the figures never drift from the real config:

1. **Cone-spawn curriculum** (sections 1–5) — the segment-spawn approach cone and its cosine
   curriculum schedules, from [`lsy_drone_racing/envs/segment_spawn.py`](../envs/segment_spawn.py).
2. **Reward-term shaping** (section 6+) — dense reward terms from
   [`lsy_drone_racing/rl/tasks/single_agent_racing.py`](tasks/single_agent_racing.py), with the
   coefficients the racing task actually runs with (the `RacingArgs` subclass of `Args`).

---

## Cone-spawn curriculum inspector

See how hard the approach actually gets as training progresses.

**Mental model** (from the module docstring):
- A spawn for target gate `i` is drawn from a truncated cone (frustum) whose **tip sits at the gate centre** and whose **axis points along the gate's entry side** (`-n`, the negative traversal normal).
- The cone radius grows with distance from the gate: `r(s) = s * tan(theta)`, `theta in [0, kappa*theta_max]`, `s in [d_min, d_min + kappa*(d_max - d_min)]`.
  - Close spawns are forced near the gate axis (well aligned with the opening).
  - Far spawns may be strongly off-axis → the policy must actively steer to line up.
- Two cosine knobs on separate `tau` windows drive difficulty: **`kappa`** (cone size) ramps first, **`p_start`** (true-start mixture) ramps later, and **`v0`** (initial speed through the gate) ramps *down* early.

`tau = global_step / total_timesteps` is normalised training progress in `[0, 1]`.

> The geometry figures use a **Plotly frame-slider** (drag `tau`). These are self-contained client-side animations — no kernel callback — so they render reliably in VSCode.

In [1]:
from typing import Callable

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from lsy_drone_racing.envs.segment_spawn import (
    SegmentSpawnConfig,
    kappa_schedule,
    p_start_schedule,
    v0_schedule,
)

# Gate geometry (config comment: 0.72 m outer frame, 0.40 m opening; pass-check box is 0.45 m).
GATE_OPENING = 0.40  # inner opening width/height [m]
GATE_OUTER = 0.72  # outer frame width/height [m]
GATE_PASS_BOX = 0.45  # tolerance box used by gate_passed() in race_core.py
GATE_HALF = GATE_OPENING / 2.0  # 0.20 m

cfg = SegmentSpawnConfig()
TAU_GRID = np.round(np.arange(0.0, 1.0001, 0.05), 2)  # slider stops


def _f(x: object) -> np.ndarray:
    """Jax scalar/array -> numpy."""
    return np.asarray(x)


def _tau_slider(frames: list, prefix: str = "tau = ") -> list:
    """A Plotly slider that animates between per-tau frames (redraw=True for 3D safety)."""
    steps = [
        dict(
            method="animate",
            label=f.name,
            args=[
                [f.name],
                dict(
                    mode="immediate",
                    frame=dict(duration=0, redraw=True),
                    transition=dict(duration=0),
                ),
            ],
        )
        for f in frames
    ]
    return [dict(active=0, currentvalue={"prefix": prefix}, pad={"t": 40}, steps=steps)]


print("Loaded SegmentSpawnConfig + schedules from lsy_drone_racing.envs.segment_spawn")

Loaded SegmentSpawnConfig + schedules from lsy_drone_racing.envs.segment_spawn


## 1. The parameters

All the static knobs that define the cone and its curriculum, pulled straight from `SegmentSpawnConfig`.

In [2]:
_DESC = {
    "gate_offset": "exit-point offset of predecessor gate (defines segment length) [m]",
    "d_min": "min standoff from the gate — always leave runway [m]",
    "d_max_cap": "global cap on segment length [m]",
    "theta_max": "cone half-angle at kappa=1 [rad]",
    "margin": "required horizontal clearance to any obstacle [m]",
    "z_min": "spawn altitude floor [m]",
    "z_max": "spawn altitude ceiling [m]",
    "n_candidates": "rejection-sampling budget per env",
    "a0": "(a) cone-size window start [tau]",
    "a1": "(a) cone-size window end [tau]",
    "kappa_min": "cone-size fraction at tau<=a0",
    "b0": "(b) true-start window start [tau]",
    "b1": "(b) true-start window end [tau]",
    "p_start_min": "true-start probability floor",
    "p_start_max": "true-start probability at tau>=b1",
    "c0": "(c) initial-speed window start [tau]",
    "c1": "(c) initial-speed window end [tau]",
    "v0_max": "initial speed along gate normal at tau=0 [m/s]",
}
print(f"{'param':>14}  {'value':>8}   description")
print("-" * 78)
for k, d in _DESC.items():
    print(f"{k:>14}  {getattr(cfg, k):>8}   {d}")
print("-" * 78)
print(f"{'GATE_OPENING':>14}  {GATE_OPENING:>8}   inner gate opening (half = {GATE_HALF} m)")
print(f"{'theta_max[deg]':>14}  {np.degrees(cfg.theta_max):>8.1f}   cone half-angle at kappa=1")
print(f"{'tan(theta_max)':>14}  {np.tan(cfg.theta_max):>8.3f}   lateral spread per metre")

         param     value   description
------------------------------------------------------------------------------
   gate_offset       0.1   exit-point offset of predecessor gate (defines segment length) [m]
         d_min      0.25   min standoff from the gate — always leave runway [m]
     d_max_cap       1.0   global cap on segment length [m]
     theta_max       0.2   cone half-angle at kappa=1 [rad]
        margin       0.3   required horizontal clearance to any obstacle [m]
         z_min       0.2   spawn altitude floor [m]
         z_max       2.0   spawn altitude ceiling [m]
  n_candidates        12   rejection-sampling budget per env
            a0      0.05   (a) cone-size window start [tau]
            a1      0.85   (a) cone-size window end [tau]
     kappa_min       0.1   cone-size fraction at tau<=a0
            b0      0.25   (b) true-start window start [tau]
            b1      0.85   (b) true-start window end [tau]
   p_start_min       0.2   true-start probability

## 2. The cosine schedules vs `tau`

Three knobs, three windows. Read top-to-bottom as the training timeline:

| knob | window | from → to | effect |
|---|---|---|---|
| `kappa` (cone size) | `[a0, a1]` | `kappa_min` → `1` | **ramps first** → widens the spawn cone (discovery) |
| `p_start` (true-start mix) | `[b0, b1]` | `p_start_min` → `p_start_max` | **ramps later** → shifts toward the deployment start distribution |
| `v0` (initial speed) | `[c0, c1]` | `v0_max` → `0` | **ramps down early** → removes the through-gate momentum crutch |

The shaded band on each panel is that knob's active window.

In [3]:
taus = np.linspace(0.0, 1.0, 400)
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "kappa  (cone size)",
        "p_start  (true-start prob.)",
        "v0  (initial speed) [m/s]",
    ),
)
specs = [
    (1, _f(kappa_schedule(taus, cfg)), (cfg.a0, cfg.a1), "royalblue"),
    (2, _f(p_start_schedule(taus, cfg)), (cfg.b0, cfg.b1), "seagreen"),
    (3, _f(v0_schedule(taus, cfg)), (cfg.c0, cfg.c1), "firebrick"),
]
for row, y, (w0, w1), c in specs:
    fig.add_trace(
        go.Scatter(x=taus, y=y, mode="lines", line=dict(color=c, width=3), showlegend=False),
        row=row,
        col=1,
    )
    fig.add_vrect(x0=w0, x1=w1, fillcolor=c, opacity=0.12, line_width=0, row=row, col=1)
fig.update_xaxes(title_text="tau = global_step / total_timesteps", row=3, col=1)
fig.update_xaxes(range=[0, 1])  # tau only ever spans [0, 1]
fig.update_layout(height=720, title="Curriculum schedules", template="plotly_white")
fig.show()

## 3. Cone shape vs gate size — side view (cross-section)

The single most useful view: a slice through the cone axis. **Drag the `tau` slider.**

- **x-axis** = distance from the gate along the entry axis (`s`); the gate plane is at `x = 0` (right edge — the drone flies right→through the gate).
- **y-axis** = lateral offset from the gate axis.
- The **thick red bar at x=0** is the gate opening (±0.20 m); the thin dotted bar is the outer frame.
- The **shaded wedge** is where cone spawns can land: at distance `s` the lateral offset reaches up to `s * tan(kappa*theta_max)`, between `d_min` and `s_hi = d_min + kappa*(d_max - d_min)`.
- The marker labels the **max off-axis offset at the far end** as a multiple of the gate half-opening — the alignment the policy must correct.

Segment length is fixed at `d_max = d_max_cap` (the worst case); edit `D_MAX` below to inspect shorter segments.

In [4]:
D_MAX = cfg.d_max_cap  # segment length to visualise (<= cfg.d_max_cap)


def _side_traces(tau: float, d_max: float) -> list:
    kap = float(kappa_schedule(tau, cfg))
    th = kap * cfg.theta_max
    d_min = cfg.d_min
    s_hi = max(d_min + kap * (d_max - d_min), d_min + 1e-3)
    s = np.linspace(d_min, s_hi, 120)
    r = s * np.tan(th)
    far = s_hi * np.tan(th)
    xpoly = np.concatenate([s, s[::-1]])
    ypoly = np.concatenate([r, -r[::-1]])
    s_full = np.linspace(0.0, s_hi, 40)
    return [
        go.Scatter(
            x=xpoly,
            y=ypoly,
            fill="toself",
            mode="lines",
            line=dict(color="royalblue"),
            fillcolor="rgba(65,105,225,0.25)",
            name="spawn region",
        ),
        go.Scatter(
            x=s_full,
            y=s_full * np.tan(th),
            mode="lines",
            line=dict(color="royalblue", dash="dot", width=1),
            showlegend=False,
        ),
        go.Scatter(
            x=s_full,
            y=-s_full * np.tan(th),
            mode="lines",
            line=dict(color="royalblue", dash="dot", width=1),
            showlegend=False,
        ),
        go.Scatter(
            x=[0, 0],
            y=[-GATE_HALF, GATE_HALF],
            mode="lines",
            line=dict(color="firebrick", width=8),
            name="gate opening (0.40 m)",
        ),
        go.Scatter(
            x=[0, 0],
            y=[-GATE_OUTER / 2, GATE_OUTER / 2],
            mode="lines",
            line=dict(color="firebrick", width=2, dash="dot"),
            name="gate frame (0.72 m)",
        ),
        go.Scatter(
            x=[s_hi],
            y=[far],
            mode="markers+text",
            text=[f"  {far:.2f} m  ({far / GATE_HALF:.1f}x gate half)"],
            textposition="middle right",
            marker=dict(color="royalblue", size=8),
            name="far-end max",
        ),
    ]


frames = [go.Frame(data=_side_traces(t, D_MAX), name=f"{t:.2f}") for t in TAU_GRID]
fig = go.Figure(data=_side_traces(TAU_GRID[0], D_MAX), frames=frames)
fig.update_xaxes(
    autorange="reversed", title="distance from gate along entry axis  s  [m]", zeroline=True
)
fig.update_yaxes(scaleanchor="x", scaleratio=1, title="lateral offset  [m]")
fig.update_layout(
    height=500,
    template="plotly_white",
    title=f"Approach cone (side view), d_max={D_MAX:.2f} m",
    sliders=_tau_slider(frames),
    legend=dict(orientation="h", y=1.08),
)
fig.show()

## 4. Cone vs gate — 3D with sampled spawns

The same cone in 3D, with actual sampled spawn points (same geometry as `_sample_cone_spawn`, minus the obstacle/altitude rejection). The gate sits in the y–z plane at the origin with its normal along +x; the drone flies in the +x direction through it. Spawns populate the entry side (−x). **Drag `tau`; rotate with the mouse.**

In [5]:
N_POINTS = 500
SEED = 0


def _square3d(half: float, color: str, width: float, name: str) -> go.Scatter3d:
    return go.Scatter3d(
        x=[0, 0, 0, 0, 0],
        y=[-half, half, half, -half, -half],
        z=[-half, -half, half, half, -half],
        mode="lines",
        line=dict(color=color, width=width),
        name=name,
    )


def _cone3d_traces(tau: float, d_max: float, n: int, seed: int) -> list:
    kap = float(kappa_schedule(tau, cfg))
    d_min = cfg.d_min
    s_hi = max(d_min + kap * (d_max - d_min), d_min + 1e-3)
    rng = np.random.default_rng(seed)
    s = rng.uniform(d_min, s_hi, n)
    theta = rng.uniform(0.0, kap * cfg.theta_max, n)
    phi = rng.uniform(0.0, 2 * np.pi, n)
    radial = s * np.tan(theta)
    return [
        go.Scatter3d(
            x=-s,
            y=radial * np.cos(phi),
            z=radial * np.sin(phi),
            mode="markers",
            marker=dict(size=2, color="royalblue", opacity=0.5),
            name="sampled spawns",
        ),
        _square3d(GATE_HALF, "firebrick", 6, "gate opening"),
        _square3d(GATE_OUTER / 2, "firebrick", 2, "gate frame"),
    ]


frames = [
    go.Frame(data=_cone3d_traces(t, cfg.d_max_cap, N_POINTS, SEED), name=f"{t:.2f}")
    for t in TAU_GRID
]
fig = go.Figure(data=_cone3d_traces(TAU_GRID[0], cfg.d_max_cap, N_POINTS, SEED), frames=frames)
lim = cfg.d_max_cap * np.tan(cfg.theta_max) * 1.05
fig.update_layout(
    height=680,
    template="plotly_white",
    title="Approach cone in 3D (drag tau, rotate with mouse)",
    sliders=_tau_slider(frames),
    scene=dict(
        xaxis_title="x along axis [m]",
        yaxis_title="y [m]",
        zaxis_title="z [m]",
        yaxis=dict(range=[-lim, lim]),
        zaxis=dict(range=[-lim, lim]),
        xaxis=dict(range=[-cfg.d_max_cap * 1.05, 0.4]),
        aspectmode="data",
    ),
)
fig.show()

## 5. How hard does the approach actually get?

Difficulty proxies as a function of `tau` (for segment length `d_max = d_max_cap`):

1. **Max off-axis offset at the far end** `= s_hi * tan(kappa*theta_max)`, in units of the gate half-opening. `1.0` = a far spawn can sit exactly at the edge of the opening; `>1` = fully outside the opening cone, so the policy must steer back in.
2. **Max standoff** `s_hi` — how far back the drone can start (runway to manage).

`v0` (momentum crutch) is shown because it offsets difficulty: while `v0 > 0` the drone is *launched* through the gate; once `v0 → 0` the wide cone is fully on the policy. `p_start` shows when the deployment start distribution takes over.

In [6]:
d_max = cfg.d_max_cap
taus = np.linspace(0.0, 1.0, 400)
kap = _f(kappa_schedule(taus, cfg))
s_hi = cfg.d_min + kap * (d_max - cfg.d_min)
max_lat = s_hi * np.tan(kap * cfg.theta_max)
v0 = _f(v0_schedule(taus, cfg))
p_start = _f(p_start_schedule(taus, cfg))

fig = make_subplots(
    rows=2,
    cols=2,
    vertical_spacing=0.13,
    horizontal_spacing=0.10,
    subplot_titles=(
        "max off-axis / gate half-opening",
        "v0 initial speed (momentum crutch) [m/s]",
        "max standoff s_hi (runway) [m]",
        "p_start (frac. from true race start)",
    ),
)
fig.add_trace(
    go.Scatter(
        x=taus, y=max_lat / GATE_HALF, line=dict(color="royalblue", width=3), showlegend=False
    ),
    row=1,
    col=1,
)
fig.add_hline(y=1.0, line=dict(color="firebrick", dash="dot"), row=1, col=1)
fig.add_trace(
    go.Scatter(x=taus, y=v0, line=dict(color="firebrick", width=3), showlegend=False), row=1, col=2
)
fig.add_trace(
    go.Scatter(x=taus, y=s_hi, line=dict(color="purple", width=3), showlegend=False), row=2, col=1
)
fig.add_trace(
    go.Scatter(x=taus, y=p_start, line=dict(color="seagreen", width=3), showlegend=False),
    row=2,
    col=2,
)
fig.update_xaxes(title_text="tau", row=2, col=1)
fig.update_xaxes(title_text="tau", row=2, col=2)
fig.update_yaxes(title_text="x gate half (0.20 m)", row=1, col=1)
fig.update_xaxes(range=[0, 1])  # tau only ever spans [0, 1]
fig.update_layout(
    height=680,
    template="plotly_white",
    title=f"Approach difficulty vs training progress (d_max = {d_max:.2f} m)",
)
fig.show()

## 6. Speed penalty (reward term) — quadratic hinge

Dense per-step penalty that keeps top speed realistic so the policy learns to **brake for the two
U-turns** instead of barrelling through them. Imported **live** from
[`tasks/single_agent_racing.py`](tasks/single_agent_racing.py) (`quadratic_speed_penalty`) with the
coefficients the racing task actually runs with (the `RacingArgs` subclass of `Args`), so this
figure matches training.

It is a one-sided **quadratic hinge** above `speed_threshold` (it replaces the earlier exponential
barrier, which penalised *every* speed and so suppressed forward flight):

```
penalty(v)   = max(0, ||vel|| - speed_threshold)**2
reward_speed = -speed_coef * penalty(v)
```

- **Exactly zero below `speed_threshold`** (a true free zone — no penalty, no gradient — the drone
  races freely on the straights), then grows **quadratically** above it: the marginal cost rises with
  the overshoot.
- C^1 at the knee (value *and* gradient are 0 at the threshold), so no gradient discontinuity.
- **`speed_threshold`** (the dotted line) is where the penalty turns on; **`speed_coef`** is the overall
  weight and, since the shape is fixed, also sets how steep the wall climbs (the grey curves sweep it;
  the bold curve is the configured value).

In [ ]:
from lsy_drone_racing.rl.tasks.single_agent_racing import RacingArgs, quadratic_speed_penalty

# RacingArgs is the task-specific Args subclass that carries the racing defaults; .create() fills
# the runtime-computed sizes -> the coefficients the racing task actually runs with.
_args = RacingArgs.create()
SPEED_COEF = _args.speed_coef
SPEED_THRESHOLD = _args.speed_threshold
COEF_SWEEP = sorted({0.02, 0.05, 0.1, float(SPEED_COEF)})  # context curves + configured one
print(f"used: speed_coef={SPEED_COEF}  speed_threshold={SPEED_THRESHOLD} m/s")


def _reward_speed(speed: np.ndarray, coef: float) -> np.ndarray:
    """Reward contribution (<= 0) of the live quadratic speed hinge for a given weight."""
    return -coef * _f(quadratic_speed_penalty(speed, SPEED_THRESHOLD))


# Sample well past the threshold so the free zone and the quadratic tail are both in frame.
v = np.linspace(0.0, SPEED_THRESHOLD + 5.0, 600)

fig = go.Figure()
for coef in COEF_SWEEP:
    is_used = np.isclose(coef, SPEED_COEF)
    fig.add_trace(
        go.Scatter(
            x=v,
            y=_reward_speed(v, coef),
            mode="lines",
            line=dict(
                color="firebrick" if is_used else "lightgray",
                width=4 if is_used else 1.5,
                dash=None if is_used else "dash",
            ),
            name=f"speed_coef={coef:g}" + (" (used)" if is_used else ""),
        )
    )

# The threshold: penalty turns on here (free to the left).
fig.add_vline(
    x=SPEED_THRESHOLD,
    line=dict(color="black", dash="dot"),
    annotation_text="speed_threshold",
    annotation_position="top",
)

# Reference markers on the configured curve at and above the threshold.
fs = SPEED_THRESHOLD + np.array([0.0, 1.0, 2.0, 4.0])
fy = _reward_speed(fs, SPEED_COEF)
fig.add_trace(
    go.Scatter(
        x=fs,
        y=fy,
        mode="markers+text",
        text=[f"  {s:g} m/s: {y:.3f}" for s, y in zip(fs, fy)],
        textposition="bottom left",
        marker=dict(color="firebrick", size=9),
        showlegend=False,
    )
)

fig.update_xaxes(title="speed  ||vel||  [m/s]", range=[0, SPEED_THRESHOLD + 5.0], zeroline=True)
fig.update_yaxes(title="reward contribution  (<= 0)")
fig.update_layout(
    height=480,
    template="plotly_white",
    title=(
        f"Speed penalty — quadratic hinge  (speed_coef={SPEED_COEF}, "
        f"speed_threshold={SPEED_THRESHOLD} m/s)"
    ),
    legend=dict(orientation="h", y=-0.2),
)
fig.show()

## 7. Gate progress potential field — the swappable variants

The dense progress reward is the *increase* of a per-gate potential `Phi` (see
[`progress_variants.py`](tasks/progress_variants.py)); the per-step reward is
`progress_coef * (Phi_t - Phi_{t-1})`. Because the target gate switches **at the gate plane**, the
telescoped reward over a whole gate segment equals `progress_coef * (Phi_at_switch - Phi_at_segment_start)`.
So **where `Phi` peaks relative to the gate plane decides whether racing forward through a gate earns
net-positive or net-negative progress.**

The variant is selected by `RacingArgs.progress = (name, coef)`; per-variant shape params live in
`RacingArgs.progress_params` (which always carries *every* variant's knobs). All three shipped
variants are imported **live** from the registry below, so the figures always match what training
runs:

- **champion** (default) — `Phi = -gate_opening_distance`: the metres-closed reward (Kaufmann et al.
  2023). A constant-gradient cone straight into the opening; symmetric entry/exit (direction is left
  to `gate_passed` + the crash penalty). `Phi <= 0`, unbounded — note this when reading the shared
  colour scales below.
- **asymmetric** — a non-negative through-gate funnel (`Phi` in (0, 1]) that **peaks at the gate
  plane** and decays *faster* on the already-passed (exit) side: `along` is inflated by `exit_scale`
  on the exit side before feeding two exponentials (`reach` long-range pull + `sharpness` near-gate
  funnel). This is the "PROPOSED (exit_scale)" field from the earlier exploration, now a first-class
  variant.
- **fancy** — a blended angle/distance potential adapted from an earlier brax reward: an exponential
  distance bump near the opening that hands off to a through-gate **alignment** term
  (`angle_progress`, +1 lined up to cross the right way, -1 on the wrong side) as the drone nears the
  plane (`gamma_angle = exp(-2*distance)`). Adapted to this repo's gate convention (traversal normal =
  gate +x axis, opening spanned by y/z).

### Proposed DISJOINT potential — the math

**Gate-local frame.** For drone position $p$ and the target gate (centre $g$, rotation $R=[\hat n\,\ \hat u\,\ \hat v]$
with $\hat n$ the traversal normal and $\hat u,\hat v$ spanning the opening), the local offset is

$$\ell = R^{\top}(p-g) = (\ell_x,\ \ell_y,\ \ell_z),$$

so $\ell_x$ is the **along-axis** coordinate (entry side $\ell_x<0$, exit side $\ell_x>0$) and $\ell_y,\ell_z$
are the lateral offsets across the opening.

**Box (corridor) distance** — lateral offsets clamped to the opening half-extent $h$, giving a rectangular
prism of equally-good crossing points:

$$o_y=\max(|\ell_y|-h,\,0),\qquad o_z=\max(|\ell_z|-h,\,0),$$
$$d=\sqrt{\ell_x^{2}+o_y^{2}+o_z^{2}},\qquad d_\perp=\sqrt{o_y^{2}+o_z^{2}}.$$

**Fly-through term** (two length scales $\rho$=`reach`, $\lambda$=`sharpness`; maximal at the plane $d=0$):

$$\Phi_{\text{box}}=\tfrac12 e^{-d/\rho}+\tfrac12 e^{-d/\lambda}.$$

**Directional funnel** ($\alpha=1$ on the entry axis, $0$ on the exit axis):

$$\alpha=\frac{1}{\pi}\arccos\!\Big(\frac{\ell_x}{d}\Big)\in[0,1].$$

**Gates** — lateral alignment $m_\perp$ (corridor half-width $\sigma$) and front-ness $f$ (centre $c$,
width $w$; stays $\approx 1$ through the plane, fades to $0$ behind):

$$m_\perp=\exp\!\Big(-\frac{d_\perp^{2}}{\sigma^{2}}\Big),\qquad
f=\tfrac12\Big(1-\tanh\frac{\ell_x-c}{w}\Big),\qquad m=m_\perp\,f.$$

**Potential and per-step reward:**

$$\boxed{\;\Phi=\Phi_{\text{box}}\,\big(m+(1-m)\,\alpha\big)\;},\qquad
r_{\text{prog}}=\kappa_{\text{prog}}\,(\Phi_t-\Phi_{t-1}).$$

**Reading it.** Where the drone is *aligned and in front* ($m\to 1$): $\Phi\to\Phi_{\text{box}}$ — the
rectangular corridor that peaks at the plane. *Off-axis or behind* ($m\to 0$): $\Phi\to\Phi_{\text{box}}\,\alpha$
— the angle funnel sets the direction (route to the front) while $\Phi_{\text{box}}$ supplies only a distance
envelope. Because $m+(1-m)\alpha\in[0,1]$, always $\Phi\le\Phi_{\text{box}}$ (no off-axis hump), and
$\Phi\in(0,1]$. Knobs: $\rho,\lambda$ (live from config), $\sigma,c,w$ (candidate).


In [8]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from lsy_drone_racing.rl.tasks.progress_variants import (
    GATE_HALF_EXTENT,
    PROGRESS_VARIANTS,
    build_progress_potential,
)
from lsy_drone_racing.rl.tasks.single_agent_racing import RacingArgs

_args = RacingArgs()
HALF = GATE_HALF_EXTENT
PROGRESS_PARAMS = _args.progress_params  # every variant's shape params (always present)
ACTIVE_VARIANT, PROGRESS_COEF = _args.progress  # the (variant, coef) training currently runs

print(f"half_extent={HALF}  active={ACTIVE_VARIANT}  progress_coef={PROGRESS_COEF}")
print("variants:", ", ".join(sorted(PROGRESS_VARIANTS)))
print("params:  ", PROGRESS_PARAMS)


def phi_variant(name: str):
    """Return f(world_xyz (N,3), gate_pos (3,), gate_quat (4,)) -> Phi (N,) for a registered variant.

    A live wrapper around ``build_progress_potential`` using the configured ``progress_params``, so
    every figure plots exactly the potential whose per-step *increase* is the progress reward training
    optimizes -- the gallery can never drift from the code.
    """
    potential = build_progress_potential(name, PROGRESS_PARAMS)

    def f(drone_xyz: np.ndarray, gate_pos: np.ndarray, gate_quat: np.ndarray) -> np.ndarray:
        N = drone_xyz.shape[0]
        dp = jnp.asarray(drone_xyz, dtype=jnp.float32).reshape(N, 1, 3)
        gp = jnp.broadcast_to(jnp.asarray(gate_pos, dtype=jnp.float32), (N, 1, 3))
        gq = jnp.broadcast_to(jnp.asarray(gate_quat, dtype=jnp.float32), (N, 1, 4))
        tg = jnp.zeros((N, 1), dtype=jnp.int32)
        return np.asarray(potential(dp, gp, gq, tg, HALF)).reshape(N)

    return f


# Common registry so every figure below stays in sync: one (name, f) entry per shipped variant, in
# registry order. The gallery auto-updates as variants are added to PROGRESS_VARIANTS.
#
# NOTE the variants live on different scales -- ``champion`` is Phi = -gate_opening_distance (<= 0,
# unbounded), while ``asymmetric`` and ``fancy`` are bounded funnels -- so absolute heights are not
# directly comparable across panels. Read each field's *shape*: where it peaks relative to the gate
# plane (x=0) is what decides whether a forward traversal banks net-positive progress per gate. The
# 7a/7c colour ranges below (ZRANGE) were tuned for (0,1] fields; champion's negative range will
# saturate there -- adjust ZRANGE per field if you want champion's well in full contrast.
FIELDS = [(name, phi_variant(name)) for name in PROGRESS_VARIANTS]
print("ready:", ", ".join(n for n, _ in FIELDS))

half_extent=0.225  active=fancy  progress_coef=5.0
variants: asymmetric, champion, fancy
params:   {'champion': {}, 'asymmetric': {'reach': 2.0, 'sharpness': 0.3, 'exit_scale': 3.0}, 'fancy': {}}
ready: champion, asymmetric, fancy


### 7a. Single-gate field (gate-local frame)

A slice through one gate at lateral height 0. The gate sits at `x = 0` with its normal along `+x`
(the drone flies left -> right through it). Entry side is `x < 0`, exit side `x > 0`. Watch where the
bright ridge (the `Phi` maximum) sits: **entry side for CURRENT, on the plane for PROPOSED.**


In [9]:
GATE_POS0 = np.zeros(3, dtype=np.float32)
GATE_QUAT0 = np.array([0.0, 0.0, 0.0, 1.0], dtype=np.float32)  # identity -> normal +x

ax = np.linspace(-3.0, 1.0, 240)  # along the gate normal; gate plane at 0
lat = np.linspace(-1.5, 1.5, 200)  # gate-local lateral y
AX, LAT = np.meshgrid(ax, lat)
pts = np.stack([AX.ravel(), LAT.ravel(), np.zeros(AX.size)], axis=1).astype(np.float32)

Zs = [f(pts, GATE_POS0, GATE_QUAT0).reshape(AX.shape) for _, f in FIELDS]
# Per-field colour ranges: CURRENT has a negative well -> [-1, 1]; PROPOSED/DISJOINT are (0, 1] ->
# [0, 1] so their corridor/funnel isn't washed out into the mid-scale. Two stacked colorbars keep
# it honest (same colour can mean different Phi across panels).
ZRANGE = [(-1.0, 1.0), (0.0, 1.0), (0.0, 1.0)]
CBARS = [
    dict(title="Phi [-1,1]", x=1.02, y=0.78, len=0.44, thickness=12),
    None,
    dict(title="Phi [0,1]", x=1.02, y=0.26, len=0.44, thickness=12),
]

fig = make_subplots(
    rows=1,
    cols=3,
    shared_yaxes=True,
    horizontal_spacing=0.05,
    subplot_titles=[t for t, _ in FIELDS],
)
for col, (Z, (zmn, zmx), cbar) in enumerate(zip(Zs, ZRANGE, CBARS), start=1):
    fig.add_trace(
        go.Heatmap(
            x=ax,
            y=lat,
            z=Z,
            zmin=zmn,
            zmax=zmx,
            colorscale="Viridis",
            showscale=cbar is not None,
            colorbar=cbar,
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Contour(
            x=ax,
            y=lat,
            z=Z,
            showscale=False,
            contours_coloring="lines",
            line=dict(color="white", width=1),
            ncontours=16,
            hoverinfo="skip",
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=[0, 0],
            y=[-HALF, HALF],
            mode="lines",
            line=dict(color="firebrick", width=7),
            showlegend=False,
        ),
        row=1,
        col=col,
    )
    fig.update_xaxes(title_text="along x [m]  (entry<0, exit>0)", row=1, col=col)
fig.update_yaxes(title_text="lateral y [m]", row=1, col=1)
fig.update_layout(
    height=440,
    template="plotly_white",
    title="Single-gate potential field (red bar = gate opening at x=0)",
)
fig.show()

### 7b. Axial slice — the whole story in one line

`Phi` along the gate axis (lateral 0), all three fields overlaid. The gate switches at `x = 0`, and on
the crossing step the reward evaluates the *post-step* potential **just past the plane**. So the
progress a forward traversal banks is `progress_coef * (Phi(just-exited) - Phi(segment-start))` — the
box quotes it per field. Watch the peak markers: CURRENT peaks on the entry side (ends below its peak
-> net-negative); PROPOSED and DISJOINT peak on the plane (end at/near the peak -> net-positive).


In [10]:
along = np.linspace(-3.0, 1.0, 600)
pax = np.stack([along, np.zeros_like(along), np.zeros_like(along)], axis=1).astype(np.float32)
COLORS = ["crimson", "seagreen", "royalblue"]

fig = go.Figure()
for (name, f), c in zip(FIELDS, COLORS):
    y = f(pax, GATE_POS0, GATE_QUAT0)
    label = name.split("  ")[0]
    if "(live)" not in name:
        label += f"  ({name.split('  ')[-1]})"
    fig.add_trace(go.Scatter(x=along, y=y, mode="lines", line=dict(color=c, width=3), name=label))
    i = int(np.argmax(y))
    fig.add_trace(
        go.Scatter(
            x=[along[i]], y=[y[i]], mode="markers", marker=dict(color=c, size=9), showlegend=False
        )
    )
fig.add_vline(x=0.0, line=dict(color="firebrick", dash="dot"))
fig.add_annotation(
    x=0.0,
    y=1.02,
    xref="x",
    yref="paper",
    text="gate plane (target switches here)",
    showarrow=False,
    font=dict(color="firebrick"),
)

# Net progress a forward traversal actually books: segment start (far entry) -> just past the plane
# (where the reward evaluates pot_curr on the crossing step). Small +x avoids the cliff at exactly
# 0.
X_START, X_EXIT = -2.8, 0.1


def _phi_at(x: float, fn: Callable) -> float:
    return float(fn(np.array([[x, 0.0, 0.0]], dtype=np.float32), GATE_POS0, GATE_QUAT0)[0])


net_rows = "<br>".join(
    f"{name.split()[0]:9s} = {PROGRESS_COEF * (_phi_at(X_EXIT, f) - _phi_at(X_START, f)):+.2f}"
    for name, f in FIELDS
)
fig.add_annotation(
    x=0.02,
    y=0.07,
    xref="paper",
    yref="paper",
    showarrow=False,
    align="left",
    font=dict(family="monospace"),
    text=(
        f"net progress over a forward traversal "
        f"(x={X_START:.1f} -> just-exited x={X_EXIT:+.1f}, "
        f"x progress_coef={PROGRESS_COEF}):<br>" + net_rows
    ),
    bgcolor="rgba(255,255,255,0.78)",
)
fig.update_layout(
    height=460,
    template="plotly_white",
    title="Axial slice (lateral=0): potential vs along-axis",
    xaxis_title="along gate normal x [m]  (entry x<0, exit x>0)",
    yaxis_title="Phi",
    legend=dict(orientation="h", y=-0.25),
)
fig.show()

### 7c. Potential field over the level0 track

The field toward each gate across the whole `level0.toml` footprint, all three fields side by side.
**Slide the target gate.** Each panel is a horizontal slice at *that gate's* height `z` (the potential
is fully 3-D; the slice is most meaningful near the selected gate). Red bars are gate openings, black
crosses obstacles, gold star the start. Look for CURRENT's dark exit-side trough behind each gate vs
the clean ridges of PROPOSED/DISJOINT — and check that DISJOINT's off-axis gradients still pull
cleanly toward each opening (no bright hump away from the corridor).


In [11]:
from pathlib import Path

from scipy.spatial.transform import Rotation as Rsp

import lsy_drone_racing
from lsy_drone_racing.envs.utils import load_track
from lsy_drone_racing.utils import load_config

ROOT = Path(lsy_drone_racing.__file__).resolve().parents[1]
_cfg = load_config(ROOT / "config" / "level0.toml")
_gates, _obstacles, _drones = load_track(_cfg.env.track)
gp_all = np.asarray(_gates["pos"], np.float32)
gq_all = np.asarray(_gates["quat"], np.float32)
obs_all = np.asarray(_obstacles["pos"], np.float32)
start = np.asarray(_drones["pos"], np.float32)[0]
G = gp_all.shape[0]
laterals = Rsp.from_quat(gq_all).as_matrix()[:, :, 1]  # gate-local y in world (G,3)
OPEN_HALF = 0.20

xs = np.linspace(-2.5, 2.5, 220)
ys = np.linspace(-1.5, 1.5, 150)
XX, YY = np.meshgrid(xs, ys)
AXES = [("x", "y"), ("x2", "y2"), ("x3", "y3")]


def _field(g: int, fn: Callable) -> np.ndarray:
    z = float(gp_all[g, 2])
    pts = np.stack([XX.ravel(), YY.ravel(), np.full(XX.size, z, np.float32)], axis=1).astype(
        np.float32
    )
    return fn(pts, gp_all[g], gq_all[g]).reshape(XX.shape)


# Per-field colour ranges (CURRENT has a well -> [-1,1]; PROPOSED/DISJOINT (0,1] -> [0,1] so the
# corridor/funnel keeps full contrast). Two stacked colorbars on the right keep it honest.
ZRANGE = [(-1.0, 1.0), (0.0, 1.0), (0.0, 1.0)]
CBARS = [
    dict(title="Phi [-1,1]", x=1.02, y=0.78, len=0.44, thickness=12),
    None,
    dict(title="Phi [0,1]", x=1.02, y=0.26, len=0.44, thickness=12),
]


def _heat(
    g: int, fn: Callable, idx: int, xa: str | None = None, ya: str | None = None
) -> go.Heatmap:
    zmn, zmx = ZRANGE[idx]
    h = go.Heatmap(
        x=xs,
        y=ys,
        z=_field(g, fn),
        colorscale="Viridis",
        zmin=zmn,
        zmax=zmx,
        showscale=CBARS[idx] is not None,
        colorbar=CBARS[idx],
    )
    if xa is not None:
        h.update(xaxis=xa, yaxis=ya)
    return h


# Gate opening segments (same every frame).
xseg, yseg = [], []
for i in range(G):
    a = gp_all[i, :2] - OPEN_HALF * laterals[i, :2]
    b = gp_all[i, :2] + OPEN_HALF * laterals[i, :2]
    xseg += [float(a[0]), float(b[0]), None]
    yseg += [float(a[1]), float(b[1]), None]

frames = [
    go.Frame(name=f"gate {g}", data=[_heat(g, fn, i, *AXES[i]) for i, (_, fn) in enumerate(FIELDS)])
    for g in range(G)
]

fig = make_subplots(
    rows=1,
    cols=3,
    shared_yaxes=True,
    horizontal_spacing=0.05,
    subplot_titles=[t for t, _ in FIELDS],
)
for i, (_, fn) in enumerate(FIELDS):
    fig.add_trace(_heat(0, fn, i), row=1, col=i + 1)
for col in (1, 2, 3):
    fig.add_trace(
        go.Scatter(
            x=xseg,
            y=yseg,
            mode="lines",
            line=dict(color="red", width=4),
            name="gate openings",
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=obs_all[:, 0],
            y=obs_all[:, 1],
            mode="markers",
            marker=dict(color="black", size=8, symbol="x"),
            name="obstacles",
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=[start[0]],
            y=[start[1]],
            mode="markers",
            marker=dict(color="gold", size=13, symbol="star", line=dict(color="black", width=1)),
            name="start",
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=gp_all[:, 0],
            y=gp_all[:, 1],
            mode="text",
            text=[str(i) for i in range(G)],
            textfont=dict(color="white", size=13),
            showlegend=False,
        ),
        row=1,
        col=col,
    )
fig.frames = frames

steps = [
    dict(
        method="animate",
        label=f.name,
        args=[
            [f.name],
            dict(
                mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0)
            ),
        ],
    )
    for f in frames
]
for col in (1, 2, 3):
    fig.update_xaxes(title_text="x [m]", row=1, col=col)
fig.update_yaxes(title_text="y [m]", row=1, col=1)
fig.update_layout(
    height=460,
    template="plotly_white",
    title="Potential field over level0 (slide target gate)",
    sliders=[dict(active=0, currentvalue={"prefix": "target "}, pad={"t": 40}, steps=steps)],
    legend=dict(orientation="h", y=-0.2),
)
fig.show()